## Table maintenance — OPTIMIZE / VACUUM / Liquid Clustering vs ZORDER

**ZORDER** co-locates rows by given columns at each `OPTIMIZE` call - effective, but it's a one-off rewrite you have to remember to rerun, and it doesn't adapt as new data lands.

**Liquid Clustering** (current recommendation for new tables) maintains clustering incrementally as data is written, replaces both classic partitioning and ZORDER, and the clustering keys can be changed later without rewriting the whole table.

**Classic partitioning** only pays off on very large tables filtered by a low-cardinality column matching the partition key; over-partitioning creates a small-file problem.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "lena066636_silver")
dbutils.widgets.text("silver_table", "orders")
dbutils.widgets.text("cluster_by_columns", "customer")


In [0]:
silver_table = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('silver_schema')}.{dbutils.widgets.get('silver_table')}"
cluster_by_columns = dbutils.widgets.get("cluster_by_columns")


In [0]:
spark.sql(f"ALTER TABLE {silver_table} CLUSTER BY ({cluster_by_columns})")
spark.sql(f"OPTIMIZE {silver_table}")


In [0]:
spark.sql(f"VACUUM {silver_table} RETAIN 168 HOURS")
